# 🦀 Rust Interview MCQ — Part 10: Design Patterns, Testing & Ecosystem (Q901–Q1000)


### Q901. In the Builder pattern, why do builder methods typically take and return `self` by value rather than `&mut self`?

- A) So the builder can be consumed and ownership transferred to the final `build()` method
- B) Because Rust doesn't allow mutable references in method chains
- C) To enable Copy semantics on the builder
- D) It's required for the typestate pattern

<details>
<summary>Show Answer</summary>

**Q901. A)** Builder methods return `self` by value to enable method chaining. The final `build()` consumes the builder and returns the constructed type. Using `&mut self` would require `build(&mut self)` and the builder would remain after build, which is often undesirable.

</details>


### Q902. What is the primary purpose of the newtype pattern (e.g., `struct Meters(f64)`)?

- A) To add methods to external types
- B) To create distinct types with the same representation for type safety and to implement traits
- C) To reduce memory usage
- D) To enable Copy for the inner type

<details>
<summary>Show Answer</summary>

**Q902. B)** Newtype wraps a type to create a distinct type. You can implement traits (including From/Into) for the newtype, add methods, and prevent mixing values (e.g., Meters vs Feet). Zero-cost abstraction—same layout as inner type.

</details>


### Q903. In the typestate pattern, how do you typically prevent invalid state transitions at compile time?

- A) Using runtime checks in every method
- B) By making invalid states unrepresentable—each state is a different type, and methods consume/transform types
- C) Using #[deny(dead_code)]
- D) By wrapping everything in Option

<details>
<summary>Show Answer</summary>

**Q903. B)** Typestate encodes state as types. E.g., `Connection<Open>` vs `Connection<Closed>`. Methods like `send()` take `Connection<Open>` and return `Connection<Open>`. You cannot call `send` on `Connection<Closed>`—the type system forbids it.

</details>


### Q904. Given this builder, what happens if you call `build()` without setting `name`?

```rust
struct User { name: String, age: u32 }
struct UserBuilder { name: Option<String>, age: u32 }
impl UserBuilder {
  fn name(mut self, n: impl Into<String>) -> Self { self.name = Some(n.into()); self }
  fn build(self) -> Result<User, &'static str> {
    Ok(User { name: self.name.ok_or("name required")?, age: self.age })
  }
}
```

- A) Compilation error
- B) Returns Err("name required") at runtime
- C) Panics
- D) Uses empty string as default

<details>
<summary>Show Answer</summary>

**Q904. B)** The builder uses Option<String> for name. build() calls ok_or("name required")? which returns Err if name is None. So calling build() without name yields Err at runtime. Compile-time enforcement would require typestate (different builder types).

</details>


### Q905. Why might you use `impl Into<String>` instead of `String` in a builder's `name()` method?

- A) Into is faster
- B) To accept both &str and String without requiring the caller to call .to_string()
- C) Into is required for builders
- D) To enable Copy

<details>
<summary>Show Answer</summary>

**Q905. B)** impl Into<String> accepts any type that implements Into<String>: String, &str, Cow<str>, etc. Callers can pass "hello" or String::from("hello") without explicit conversion. More ergonomic API.

</details>


### Q906. In a typestate API, `fn close(self) -> Connection<Closed>` consumes `Connection<Open>`. What prevents calling `send()` after `close()`?

- A) Runtime panic
- B) The Connection<Closed> type has no send method; only Connection<Open> does
- C) Drop implementation
- D) The compiler inserts runtime checks

<details>
<summary>Show Answer</summary>

**Q906. B)** Typestate encodes state in the type. Connection<Closed> is a different type from Connection<Open>. impl blocks provide send() only for Connection<Open>. After close(), you have Connection<Closed>—send() doesn't exist. Compile-time guarantee.

</details>


### Q907. What does the newtype pattern give you that a type alias does not?

- A) Smaller binary size
- B) A distinct type that can have its own trait implementations and prevents mixing with the inner type
- C) Faster compilation
- D) Automatic Deref to the inner type

<details>
<summary>Show Answer</summary>

**Q907. B)** type Meters = f64 is just an alias—Meters and f64 are interchangeable. struct Meters(f64) is a new type. You can impl traits for Meters, add methods, and the type checker prevents passing f64 where Meters is expected.

</details>


### Q908. For a builder with optional fields, which approach gives compile-time enforcement that required fields are set?

- A) Using Option and checking at build() time
- B) Typestate: separate builder types for each required field (Builder0, Builder1, etc.), where build() only exists on the final state
- C) Using default values
- D) Using #[must_use]

<details>
<summary>Show Answer</summary>

**Q908. B)** Typestate builders use different types: Builder, BuilderWithName, BuilderWithNameAndAge. build() is only on BuilderWithNameAndAge. You cannot call build() until you've chained through all required setters. Compile-time guarantee.

</details>


### Q909. When using `#[derive(Clone)]` on a newtype, what gets cloned?

- A) Only the newtype wrapper
- B) The inner value (Clone is derived to clone the inner type)
- C) Nothing—Clone cannot be derived for newtypes
- D) A shallow copy of the pointer

<details>
<summary>Show Answer</summary>

**Q909. B)** derive(Clone) on struct Foo(T) generates impl Clone for Foo that clones the inner T. So Foo(x).clone() gives Foo(x.clone()). The newtype is transparent for Clone.

</details>


### Q910. In a typestate state machine, why is `fn transition(self) -> StateB` preferred over `fn transition(&mut self)`?

- A) It's faster
- B) Consuming self ensures the old state is no longer accessible; you cannot accidentally use the old state after transitioning
- C) &mut self would require Clone
- D) Rust requires it for typestate

<details>
<summary>Show Answer</summary>

**Q910. B)** Taking self by value consumes the old state. The caller receives StateB and the old StateA is gone. With &mut self, you'd mutate in place—both states could be accessible or you'd need to track manually. Consuming self makes invalid states unrepresentable.

</details>


### Q911. What happens when a MutexGuard goes out of scope?

- A) The mutex is locked again
- B) The guard's Drop implementation releases the lock
- C) A panic occurs
- D) The mutex is destroyed

<details>
<summary>Show Answer</summary>

**Q911. B)** MutexGuard implements Drop. When the guard is dropped (goes out of scope, or explicitly with drop(guard)), Drop::drop runs and releases the lock. This is RAII—resource acquisition is initialization, release is destruction.

</details>


### Q912. Why can't you store a MutexGuard in a struct and return that struct from a function?

- A) MutexGuard is not Send
- B) MutexGuard holds a reference/lifetime tied to the Mutex; returning it would require the struct to outlive the Mutex, which may not be expressible
- C) MutexGuard is Copy
- D) MutexGuard cannot be dropped

<details>
<summary>Show Answer</summary>

**Q912. B)** MutexGuard<'a> has a lifetime tying it to the Mutex. When you return a struct containing it, the struct's lifetime must not exceed the Mutex. Often the Mutex is local—you can't return a guard that outlives it. The guard must be dropped before the Mutex is dropped.

</details>


### Q913. What is the relationship between RAII and the Drop trait?

- A) They are unrelated
- B) Drop is the Rust mechanism that implements RAII—cleanup runs when a value goes out of scope
- C) RAII replaces Drop
- D) Drop is only for async code

<details>
<summary>Show Answer</summary>

**Q913. B)** RAII = Resource Acquisition Is Initialization. In Rust, Drop::drop is called when a value goes out of scope. So acquiring a resource (e.g., lock) in the constructor and releasing it in Drop implements RAII. MutexGuard, File, etc. use this.

</details>


### Q914. Why does RwLockReadGuard need to be held for the duration of the read?

- A) To prevent the RwLock from being dropped
- B) The guard ensures the lock is held; dropping it releases the read lock. Holding it prevents writers from acquiring the write lock
- C) To satisfy Send
- D) To enable Sync

<details>
<summary>Show Answer</summary>

**Q914. B)** RwLockReadGuard holds the read lock. While held, other readers can acquire, but writers block. When the guard is dropped, the read lock is released. You must hold it for the entire duration you're reading to prevent data races.

</details>


### Q915. What can go wrong if you implement Drop and also use #[may_dangle] (dropck) incorrectly?

- A) Nothing—#[may_dangle] is safe
- B) Use-after-free: Drop may access borrowed data that was already dropped
- C) Double free
- D) Stack overflow

<details>
<summary>Show Answer</summary>

**Q915. B)** dropck (drop check) ensures Drop doesn't access dangling references. #[may_dangle] opts out of the check for a lifetime. If you use it incorrectly, Drop might run and access data that was already dropped—use-after-free, UB.

</details>


### Q916. In `let guard = mutex.lock().unwrap();`, when is the lock released?

- A) Immediately after the statement
- B) When guard goes out of scope (at the end of the block or when dropped)
- C) When unwrap() returns
- D) Never—unwrap consumes it

<details>
<summary>Show Answer</summary>

**Q916. B)** lock() returns LockResult<MutexGuard>. unwrap() gives MutexGuard. The guard holds the lock. When guard goes out of scope, its Drop runs and releases the lock. The scope of guard determines how long the lock is held.

</details>


### Q917. Why might you use scopeguard or similar instead of implementing Drop for simple cleanup?

- A) scopeguard is faster
- B) For one-off cleanup without defining a new type; you can run a closure on scope exit
- C) Drop cannot run cleanup
- D) scopeguard is required for locks

<details>
<summary>Show Answer</summary>

**Q917. B)** scopeguard::defer(|| cleanup()) runs the closure when the scope ends. Useful when you don't want to create a wrapper type just for Drop. Also supports different behaviors (always run vs. run on success/failure).

</details>


### Q918. What happens if Drop::drop panics?

- A) The program aborts (double panic)
- B) The panic is ignored
- C) The rest of Drop runs
- D) Undefined behavior

<details>
<summary>Show Answer</summary>

**Q918. A)** If a panic is already in progress and Drop panics, Rust aborts (double panic). Panicking in Drop is generally discouraged—it can cause aborts and makes cleanup unpredictable.

</details>


### Q919. How does RefCell differ from Mutex in terms of RAII?

- A) RefCell doesn't use RAII
- B) RefCell's Ref/RefMut guards implement RAII—they release the borrow when dropped. Mutex uses MutexGuard; both use guard types
- C) RefCell has no guards
- D) Mutex doesn't use RAII

<details>
<summary>Show Answer</summary>

**Q919. B)** Both use guard types. MutexGuard holds the lock; Ref and RefMut hold the RefCell borrow. When the guard is dropped, the lock/borrow is released. RefCell is single-threaded; Mutex is for multi-threaded. Both are RAII.

</details>


### Q920. Why is it problematic to have a type that implements both Drop and Copy?

- A) Rust forbids it
- B) Copy creates duplicates without running Drop on the 'original'; when each copy is dropped, Drop runs multiple times on logically the same resource—double free or similar
- C) Copy is slower with Drop
- D) Drop cannot be implemented for Copy types

<details>
<summary>Show Answer</summary>

**Q920. B)** Copy types are implicitly duplicated. If the type owns a resource (file handle, allocation), each copy thinks it owns it. When each is dropped, Drop runs multiple times—double free, double close. Copy should only be for types that don't own resources.

</details>


### Q921. Where does Cargo look for integration tests by default?

- A) In src/
- B) In tests/ directory at the crate root
- C) In test/
- D) Alongside the source files

<details>
<summary>Show Answer</summary>

**Q921. B)** Integration tests go in tests/ at the crate root. Each .rs file is compiled as a separate crate that links your library. Unit tests go in the same file with #[cfg(test)] or in a tests module.

</details>


### Q922. What does #[cfg(test)] do?

- A) Runs the test
- B) Includes the item only when compiling for tests (cargo test)
- C) Marks the item as a test function
- D) Enables test features in dependencies

<details>
<summary>Show Answer</summary>

**Q922. B)** #[cfg(test)] means the item is compiled only when cfg(test) is true—i.e., when running cargo test. Used for test-only modules, helpers, or code that shouldn't be in the release binary.

</details>


### Q923. What is the difference between #[test] and a regular function in a #[cfg(test)] module?

- A) No difference
- B) #[test] marks a function as a test case; Cargo runs it when you run cargo test. A regular function is just a helper
- C) #[test] is for integration tests only
- D) Regular functions cannot be in #[cfg(test)]

<details>
<summary>Show Answer</summary>

**Q923. B)** #[test] marks a function as a test. cargo test discovers and runs them. Regular functions in the same module are test helpers—they run only when called by a test. Both are compiled only with #[cfg(test)].

</details>


### Q924. What does #[should_panic] do?

- A) Makes the test panic
- B) The test passes only if it panics; if it doesn't panic, the test fails
- C) Catches panics
- D) Skips the test

<details>
<summary>Show Answer</summary>

**Q924. B)** #[should_panic] inverts success: the test passes if it panics. If the test completes without panicking, it fails. Used for testing that invalid inputs or states correctly cause a panic. Can add expected = "substring" to check panic message.

</details>


### Q925. How do you run a specific test by name?

- A) cargo test test_name
- B) cargo test <pattern> — runs tests whose names contain the pattern
- C) cargo test --name test_name
- D) You cannot run a single test

<details>
<summary>Show Answer</summary>

**Q925. B)** cargo test foo runs all tests whose names contain 'foo'. cargo test test_module::test_fn runs that specific test. The argument is a filter pattern, not an exact match.

</details>


### Q926. In integration tests, how do you access your library's public API?

- A) Use crate::
- B) Add your crate as a dependency; each test file is a separate crate, so use `use your_crate_name::...`
- C) Use super::
- D) Integration tests cannot use the library

<details>
<summary>Show Answer</summary>

**Q926. B)** Integration tests are separate crates. In Cargo.toml they get your lib as a dev-dependency. In the test file, you use `use your_crate::function` or `your_crate::module::item`. They only see the public API.

</details>


### Q927. What does #[ignore] do on a test?

- A) Fails the test
- B) The test is not run by default; use cargo test --ignored to run it
- C) Hides the test
- D) Marks it as flaky

<details>
<summary>Show Answer</summary>

**Q927. B)** #[ignore] skips the test in normal cargo test. Use cargo test --ignored to run only ignored tests, or cargo test --include-ignored to run all. Useful for slow or environment-specific tests.

</details>


### Q928. Why might you put test helpers in a common module (e.g., tests/common/mod.rs)?

- A) To make them run first
- B) Cargo treats files in tests/ as separate test crates; common/mod.rs is not compiled as a test, so you can share helpers across test files by including it
- C) To reduce compile time
- D) common is a special Cargo directory

<details>
<summary>Show Answer</summary>

**Q928. B)** Each file in tests/ is a separate crate. To share code, use tests/common/mod.rs. Files named common*.rs are not compiled as tests. In each test file, use `mod common;` and then `common::helper()`.

</details>


### Q929. What does `cargo test --no-run` do?

- A) Runs tests without output
- B) Compiles tests but does not run them
- C) Skips tests
- D) Runs tests in parallel

<details>
<summary>Show Answer</summary>

**Q929. B)** --no-run compiles the test binary but doesn't execute it. Useful for checking that tests compile, or to get the path to the test binary for debugging (e.g., under gdb).

</details>


### Q930. In a #[test] function, what type does a Result return value indicate?

- A) The test framework ignores it
- B) Ok(()) means pass, Err means fail (the test fails with the error)
- C) Result is not allowed in tests
- D) It enables ? in tests

<details>
<summary>Show Answer</summary>

**Q930. B)** A test can return Result<(), E> where E: Debug. Ok(()) passes; Err(e) fails and prints e. This lets you use ? in tests: fn test() -> Result<(), Box<dyn Error>> { ... }.

</details>


### Q931. What is the main idea behind property-based testing (e.g., proptest)?

- A) Testing with fixed inputs
- B) Defining properties that should hold for all inputs; the framework generates random inputs and checks the property
- C) Testing only edge cases
- D) Mutation testing

<details>
<summary>Show Answer</summary>

**Q931. B)** Property-based testing: you state invariants (e.g., reverse(reverse(x)) == x). The framework generates many random inputs and verifies the property. Finds edge cases you might not have thought of. proptest and quickcheck do this in Rust.

</details>


### Q932. How does proptest typically shrink failing inputs?

- A) It doesn't shrink
- B) It tries to find a smaller input that still fails, making the failure easier to debug
- C) It expands the input
- D) Shrinking is only in quickcheck

<details>
<summary>Show Answer</summary>

**Q932. B)** When a property fails, proptest tries to shrink the input (e.g., smaller numbers, shorter vecs) while keeping the failure. The goal is a minimal failing case. Helps identify the root cause.

</details>


### Q933. What is libfuzzer (cargo-fuzz) used for?

- A) Unit testing
- B) Fuzzing: generating random inputs to find crashes, panics, or UB
- C) Benchmarking
- D) Documentation

<details>
<summary>Show Answer</summary>

**Q933. B)** Fuzzing feeds random/corpus inputs to your code to find bugs. cargo-fuzz uses libFuzzer. You write a fuzz target; the fuzzer mutates inputs to maximize coverage and find crashes. Great for parsers, decoders, etc.

</details>


### Q934. What does mockall typically provide?

- A) Property-based testing
- B) Mock implementations of traits—you define expected calls and return values for testing
- C) Fuzzing
- D) Code coverage

<details>
<summary>Show Answer</summary>

**Q934. B)** mockall generates mock implementations of traits. You set expectations (this method will be called with X, return Y). Useful for isolating units under test by replacing dependencies with mocks.

</details>


### Q935. In proptest, what does `proptest!` macro do?

- A) Defines a single test
- B) Defines a property-based test: generates inputs from the strategy and runs the test body for each
- C) Configures the test runner
- D) Enables shrinking

<details>
<summary>Show Answer</summary>

**Q935. B)** proptest! { strategy => |input| { ... } } runs the closure with many generated inputs. The strategy (e.g., 0u32..100) defines the input space. The closure must hold for all inputs (or proptest finds a failing case).

</details>


### Q936. Why might you use a fuzz corpus directory?

- A) To store test results
- B) To seed the fuzzer with interesting inputs that improve coverage and help find new paths
- C) To disable fuzzing
- D) To run unit tests

<details>
<summary>Show Answer</summary>

**Q936. B)** A corpus is a set of inputs the fuzzer has found. Storing them lets the fuzzer start from known-interesting inputs and mutate them. Improves coverage and finds bugs faster. Often committed to the repo.

</details>


### Q937. What is the main limitation of mocking (e.g., mockall) compared to integration tests?

- A) Mocks are slower
- B) Mocks test against a fake implementation; they don't catch bugs in the real dependency or integration issues
- C) Mocks cannot be used with traits
- D) Mocks require async

<details>
<summary>Show Answer</summary>

**Q937. B)** Mocks replace real dependencies. You test your code in isolation, but you don't test the real interaction. Integration tests use real dependencies and catch more bugs. Use both: mocks for unit tests, integration tests for E2E.

</details>


### Q938. In quickcheck, what trait must your type implement to be used as a random test input?

- A) Debug
- B) Arbitrary
- C) Clone
- D) Default

<details>
<summary>Show Answer</summary>

**Q938. B)** quickcheck uses the Arbitrary trait to generate random values. impl Arbitrary for MyType defines arbitrary(). proptest uses Strategy instead of Arbitrary, but the idea is similar—a way to generate random test data.

</details>


### Q939. What does `#[derive(Arbitrary)]` (in quickcheck) do?

- A) Makes the type Copy
- B) Generates an Arbitrary implementation that recursively uses Arbitrary for fields
- C) Enables fuzzing
- D) Adds Default

<details>
<summary>Show Answer</summary>

**Q939. B)** derive(Arbitrary) implements Arbitrary by generating each field with its Arbitrary impl. For enums, it picks a variant. Lets you quickly get random instances of structs for property-based tests.

</details>


### Q940. When using test fixtures (setup/teardown), what Rust pattern is commonly used?

- A) Fixtures are not possible in Rust
- B) A struct that implements Drop for teardown; the constructor does setup. Use in a scope so Drop runs
- C) Global mutable state
- D) Macros only

<details>
<summary>Show Answer</summary>

**Q940. B)** RAII again: create a fixture struct in a block. Constructor does setup (create temp file, start server). Drop does teardown (delete file, stop server). When the struct goes out of scope, cleanup runs. No special test framework needed.

</details>


### Q941. What is the main advantage of cargo-nextest over cargo test?

- A) It runs tests in a single thread
- B) Faster test runs via parallelization, test grouping, and better failure output
- C) It replaces proptest
- D) It only runs integration tests

<details>
<summary>Show Answer</summary>

**Q941. B)** cargo-nextest runs tests in parallel by default, groups related tests, and provides clearer failure output. Drop-in replacement for cargo test. Speeds up large test suites significantly.

</details>


### Q942. In a Cargo workspace, how do workspace members share the same lockfile?

- A) They don't
- B) The lockfile (Cargo.lock) is at the workspace root; all members use it
- C) Each member has its own lockfile
- D) You must run cargo update in each member

<details>
<summary>Show Answer</summary>

**Q942. B)** A workspace has a root Cargo.toml. Cargo.lock lives at the root. All workspace members resolve dependencies against this single lockfile, ensuring consistent versions across the workspace.

</details>


### Q943. What does a default feature in Cargo do?

- A) Enables the feature by default when the crate is used as a dependency
- B) Disables the feature
- C) Makes the feature required
- D) Runs at startup

<details>
<summary>Show Answer</summary>

**Q943. A)** default = ["feature1"] in [features] means when someone adds your crate, feature1 is enabled unless they opt out with default-features = false. Use for optional but commonly wanted functionality.

</details>


### Q944. What does #[cfg(feature = "foo")] do?

- A) Enables the feature
- B) Includes the item only when the "foo" feature is enabled
- C) Checks at runtime
- D) Requires the feature

<details>
<summary>Show Answer</summary>

**Q944. B)** #[cfg(feature = "foo")] compiles the item only when the crate is built with --features foo (or foo in default features). Used for optional code—e.g., serde support behind a feature.

</details>


### Q945. What is the difference between [dependencies] and [dev-dependencies]?

- A) No difference
- B) dev-dependencies are only used for tests, benchmarks, and examples; not in the main build
- C) dev-dependencies are faster
- D) dev-dependencies cannot have features

<details>
<summary>Show Answer</summary>

**Q945. B)** dev-dependencies are used when compiling tests, benches, examples. They're not linked into the main library/binary. Use for test-only crates (e.g., proptest, mockall) to keep the main build lean.

</details>


### Q946. How do you add an optional dependency that enables a feature when present?

- A) optional = true in the dependency; then list it in [features]
- B) Use [optional-dependencies]
- C) Put it in dev-dependencies
- D) Use cfg

<details>
<summary>Show Answer</summary>

**Q946. A)** serde = { version = "1.0", optional = true } and [features] serde = ["dep:serde"] or serde = ["serde"]. When the user enables your "serde" feature, the serde dependency is included.

</details>


### Q947. What does #[cfg(target_os = "windows")] do?

- A) Runs only on Windows
- B) Includes the item only when compiling for Windows target
- C) Checks at runtime
- D) Enables Windows APIs

<details>
<summary>Show Answer</summary>

**Q947. B)** cfg(target_os = "windows") is a compile-time check. The code is included only when building for Windows. For other targets, the code is not compiled at all. Used for platform-specific implementations.

</details>


### Q948. In a workspace, can a member depend on another member?

- A) No
- B) Yes, using path = "../other-member" and the member's package name
- C) Only if they're in the same directory
- D) Only for dev-dependencies

<details>
<summary>Show Answer</summary>

**Q948. B)** my-package = { path = "../other-crate" }. The workspace member is a path dependency. Cargo resolves it within the workspace. Common for splitting a project into multiple crates.

</details>


### Q949. What does cfg(all(feature = "a", not(feature = "b"))) mean?

- A) Feature a or not b
- B) Feature a is enabled AND feature b is disabled
- C) Either condition
- D) Invalid syntax

<details>
<summary>Show Answer</summary>

**Q949. B)** all(...) requires all conditions. So: a enabled and b disabled. any(...) would mean at least one. not(...) inverts. Used for mutually exclusive features or complex gating.

</details>


### Q950. What is the purpose of [patch] in Cargo.toml?

- A) To patch the Rust compiler
- B) To override a dependency with a local path or git repo (e.g., for testing a fix)
- C) To enable all features
- D) To patch features

<details>
<summary>Show Answer</summary>

**Q950. B)** [patch] lets you replace a dependency. E.g., patch crates.io version with a git branch or local path. Useful for trying upstream fixes before release, or forking a crate with local changes.

</details>


### Q951. What does cfg(debug_assertions) mean?

- A) Always true
- B) True in debug builds (dev), false in release builds
- C) True when running tests
- D) Runtime check

<details>
<summary>Show Answer</summary>

**Q951. B)** debug_assertions is enabled in dev (cargo build) and disabled in release (cargo build --release). Use for extra checks in dev that you don't want in production (e.g., debug_assert!, expensive validation).

</details>


### Q952. What does #[serde(rename = "foo")] on a struct field do?

- A) Renames the field at runtime
- B) When serializing/deserializing, use "foo" as the key instead of the Rust field name
- C) Creates a new field
- D) Only affects serialization

<details>
<summary>Show Answer</summary>

**Q952. B)** rename changes the key in the serialized format. struct Foo { #[serde(rename = "bar")] x: i32 } serializes as {"bar": 0} not {"x": 0}. Works for both serialize and deserialize.

</details>


### Q953. What is the purpose of serde's externally_tagged enum representation?

- A) No tags
- B) The enum variant is the key; the value is the variant's data. E.g. {"Foo": {"a": 1}}
- C) Internal tags only
- D) Adjacently tagged

<details>
<summary>Show Answer</summary>

**Q953. B)** externally_tagged (default for enums): variant name is the key, content is the value. {"Unit": null}, {"Tuple": [1,2]}, {"Struct": {"x":1}}. Good for JSON. internally_tagged uses a single object with a tag field.

</details>


### Q954. What does serde's #[serde(skip)] do?

- A) Skips serialization only
- B) Skips the field for both serialize and deserialize; deserialize uses Default
- C) Skips deserialization only
- D) Causes an error

<details>
<summary>Show Answer</summary>

**Q954. B)** skip omits the field. Serialize: field not written. Deserialize: field gets Default::default() (or use skip_deserializing/skip_serializing for one-way). Useful for computed or redundant fields.

</details>


### Q955. What is zero-copy deserialization (e.g., serde's borrow)?

- A) Deserialization without allocation
- B) Deserializing into borrowed references (e.g., &str) that point into the input buffer, avoiding allocation
- C) Copying the input
- D) Skipping deserialization

<details>
<summary>Show Answer</summary>

**Q955. B)** With &'a str or &'a [u8], serde can deserialize by borrowing from the input. The output references the input—no allocation. Requires the input to outlive the output. Used in serde_json::Value with borrow, or in custom Deserialize impls.

</details>


### Q956. When implementing Deserialize manually, what does the 'de lifetime in DeserializeSeed<'de> represent?

- A) The deserializer's lifetime
- B) The lifetime of the data being deserialized (input buffer); borrowed output can live at most this long
- C) The seed's lifetime
- D) The output lifetime

<details>
<summary>Show Answer</summary>

**Q956. B)** 'de is the lifetime of the input. When deserializing into &'de str, the output borrows from the input. The 'de ties the output lifetime to the input, enabling zero-copy deserialization.

</details>


### Q957. What does #[serde(flatten)] do?

- A) Flattens nested structs: fields of the inner struct appear at the same level in the serialized output
- B) Nests the struct
- C) Skips the field
- D) Renames the field

<details>
<summary>Show Answer</summary>

**Q957. A)** flatten on a field embeds its fields into the parent. struct Outer { a: i32, #[serde(flatten)] inner: Inner } with Inner { b: i32 } serializes as {"a":0,"b":0}. Useful for combining configs or API responses.

</details>


### Q958. What is the untagged enum representation?

- A) No variant info; try each variant in order until one deserializes successfully
- B) Uses a tag field
- C) Same as externally tagged
- D) Invalid

<details>
<summary>Show Answer</summary>

**Q958. A)** untagged: no variant marker. Deserializer tries each variant in order; first success wins. Useful when the data shape uniquely identifies the variant (e.g., single field vs object). Can be ambiguous.

</details>


### Q959. Why might custom Serialize/Deserialize implementations be needed instead of derive?

- A) Derive is never sufficient
- B) For non-standard formats, custom logic, or when the Rust structure doesn't map 1:1 to the format
- C) Only for performance
- D) Only for zero-copy

<details>
<summary>Show Answer</summary>

**Q959. B)** Derive works for straightforward mappings. Custom impls for: different format shapes, validation during deserialize, conditional fields, or formats serde doesn't support out of the box.

</details>


### Q960. What does #[serde(default)] on a field do?

- A) Uses Default::default() when the field is missing during deserialization
- B) Makes the field optional
- C) Only for Option
- D) Skips the field

<details>
<summary>Show Answer</summary>

**Q960. A)** default: if the key is absent in the input, use Default::default() for the field. Lets you add new optional fields without breaking old data. Can use default = "path::to::function" for custom default.

</details>


### Q961. In serde, what is the relationship between Serialize and Serializer?

- A) They are the same
- B) Serialize is implemented by your type (data); Serializer is implemented by the format (JSON, etc.). Your type calls serializer methods to emit its data
- C) Serializer is for deserialization
- D) Only Serialize exists

<details>
<summary>Show Answer</summary>

**Q961. B)** Serialize: your type implements this. It receives a Serializer and calls methods like serialize_i32, serialize_struct, etc. Serializer: the format (serde_json, etc.) implements this. It receives the calls and writes bytes. Data and format are separate.

</details>


### Q962. What does thiserror typically provide?

- A) Panic handling
- B) Derive macro for Error, Display, and From impls to create ergonomic error types
- C) Logging
- D) Result type

<details>
<summary>Show Answer</summary>

**Q962. B)** thiserror::Error derive implements std::error::Error, Display (from #[error("...")]), and From (from #[from]). Reduces boilerplate for library error types. anyhow is for applications (easy ? conversion).

</details>


### Q963. What is anyhow primarily used for?

- A) Library error types
- B) Application error handling: easy Result propagation with ? and context, without defining custom error types
- C) Panic recovery
- D) Logging errors

<details>
<summary>Show Answer</summary>

**Q963. B)** anyhow::Result<T> and anyhow::Error let you ? any Error into them. .context("msg") adds context. Good for applications where you want to bubble errors and add context without defining a custom error enum.

</details>


### Q964. When does the ? operator NOT work for propagating errors?

- A) Always works
- B) When the function return type doesn't implement From for the error type, or in main (unless main returns Result)
- C) Only in async
- D) Only for Option

<details>
<summary>Show Answer</summary>

**Q964. B)** ? requires From<E> for the function's error type. If your function returns Result<T, MyError>, ? only works for errors that impl Into<MyError>. main() can return Result for ? to work; otherwise use .unwrap() or match.

</details>


### Q965. What does #[from] in thiserror do?

- A) Marks the field as the source
- B) Generates a From impl so you can ? that error type into your error
- C) Converts to String
- D) Enables Display

<details>
<summary>Show Answer</summary>

**Q965. B)** #[from] on a variant field generates impl From<ThatError> for YourError. So ? can convert that error into your type. #[source] marks the field for Error::source(); #[from] implies #[source] and adds From.

</details>


### Q966. Why might you implement From<OtherError> for YourError?

- A) To convert your error to OtherError
- B) To allow ? to convert OtherError into YourError automatically
- C) To enable Clone
- D) For Display

<details>
<summary>Show Answer</summary>

**Q966. B)** impl From<OtherError> for YourError lets you use ? on functions that return OtherError when your function returns Result<_, YourError>. The ? will automatically convert. Reduces boilerplate at call sites.

</details>


### Q967. What is the purpose of .context() in anyhow?

- A) To catch errors
- B) To add contextual information to the error chain (e.g., "while parsing config")
- C) To log
- D) To convert to String

<details>
<summary>Show Answer</summary>

**Q967. B)** err.context("while parsing config") wraps the error with a message. When you print the error (or use {:?}), you see the chain. Helps debug: "failed to read file" -> "while parsing config" -> "invalid UTF-8".

</details>


### Q968. In a custom Error type, why implement Error::source()?

- A) It's required
- B) To allow error reporters to walk the cause chain and display the full error story
- C) For From
- D) For Display

<details>
<summary>Show Answer</summary>

**Q968. B)** source() returns the underlying error. Tools can iterate: error.source().and_then(|e| e.source())... to print the full chain. thiserror's #[source] generates this. Helps debugging and error reporting.

</details>


### Q969. What happens if you use ? on an Err in main() when main returns ()?

- A) It works
- B) Compilation error: ? can only be used in functions that return Result or Option
- C) Runtime panic
- D) The error is ignored

<details>
<summary>Show Answer</summary>

**Q969. B)** ? requires the function to return Result<T, E> (or Option, etc.). main() -> () doesn't. Use main() -> Result<(), Box<dyn Error>> or main() -> Result<(), E> to use ?. Or handle errors manually.

</details>


### Q970. When would you use Box<dyn Error> as an error type?

- A) Never
- B) When you want to accept any error type (e.g., in main or a generic handler) without defining a custom enum
- C) Only for libraries
- D) For performance

<details>
<summary>Show Answer</summary>

**Q970. B)** Result<T, Box<dyn Error>> accepts any error. ? works for any E: Into<Box<dyn Error>>. Good for applications where you don't care about the exact type. anyhow::Error is often better (context, etc.).

</details>


### Q971. What does map_err do?

- A) Maps the Ok value
- B) Maps the Err value to a different error type (e.g., for conversion before ?)
- C) Ignores the error
- D) Unwraps

<details>
<summary>Show Answer</summary>

**Q971. B)** result.map_err(|e| MyError::from(e)) converts Err(E) to Err(MyError). Used when you need to convert an error before ? (e.g., your function returns Result<T, MyError> but the inner call returns Result<T, IoError>).

</details>


### Q972. Why does `fn foo() -> Result<T, Box<dyn Error>>` allow `?` on any error type?

- A) Box<dyn Error> implements From for all types
- B) Any type implementing Error implements From for Box<dyn Error> via the blanket impl From<E: Error> for Box<dyn Error>
- C) ? converts all errors to String
- D) It doesn't—you must use map_err

<details>
<summary>Show Answer</summary>

**Q972. B)** The standard library has impl From<E> for Box<dyn Error> where E: Error. So any Error can be converted to Box<dyn Error>. The ? operator uses From for conversion, so ? works for any error type.

</details>


### Q973. What is the main difference between the log crate and the tracing crate?

- A) They are identical
- B) tracing adds spans (structured context), async support, and richer metadata; log is simpler, level-based
- C) log is for async only
- D) tracing replaces log

<details>
<summary>Show Answer</summary>

**Q973. B)** log: log::info!("msg") with levels. tracing: spans (enter/exit), structured fields, works with async. tracing can emit to log. tracing is more powerful for distributed and async systems.

</details>


### Q974. What is a span in tracing?

- A) A log level
- B) A period of time/scope with a name and optional fields; events can be associated with spans
- C) A subscriber
- D) A filter

<details>
<summary>Show Answer</summary>

**Q974. B)** A span represents a unit of work (e.g., "handle_request", "query_db"). You enter/exit spans. Events inside a span are associated with it. Subscribers can track span hierarchy and timing.

</details>


### Q975. What does tracing::instrument on a function do?

- A) Measures execution time
- B) Creates a span for the function's execution with the function name and optionally its arguments as fields
- C) Logs the return value
- D) Enables async

<details>
<summary>Show Answer</summary>

**Q975. B)** #[instrument] creates a span when the function is called. By default the span name is the function name. Arguments can be included as fields (skip to exclude). Simplifies adding tracing to functions.

</details>


### Q976. What is a tracing Subscriber?

- A) A log level
- B) The backend that receives spans and events and decides what to do (e.g., format and write to stdout)
- C) A span
- D) A filter

<details>
<summary>Show Answer</summary>

**Q976. B)** Subscriber is the trait for the consumer. tracing_subscriber provides implementations: fmt (human-readable), json, etc. You set the subscriber with set_global_default. It receives all spans and events.

</details>


### Q977. What does log::Record contain?

- A) Only the message
- B) Level, target (module path), message, and optional key-value metadata
- C) Spans
- D) Subscriber config

<details>
<summary>Show Answer</summary>

**Q977. B)** Record has level, target (e.g., "my_crate::module"), args (the message), and metadata. Loggers use this to format and output. Simpler than tracing's Event which has span context.

</details>


### Q978. Why might you use tracing's structured fields instead of string interpolation?

- A) Strings are faster
- B) Structured fields can be queried, filtered, and exported to systems that understand key-value data (e.g., OpenTelemetry, JSON logs)
- C) Only for debugging
- D) They are required

<details>
<summary>Show Answer</summary>

**Q978. B)** info!(user_id = %id, "user logged in") emits structured data. Log aggregators can filter by user_id. JSON exporters get {"user_id": 123}. Better than "user 123 logged in" for machine processing.

</details>


### Q979. What does tracing_subscriber::fmt::init() do?

- A) Initializes the log crate
- B) Sets up a default fmt subscriber and installs it as the global default for tracing
- C) Enables all levels
- D) Configures logging

<details>
<summary>Show Answer</summary>

**Q979. B)** init() creates a fmt subscriber (human-readable output to stderr) and sets it as the global subscriber. After this, tracing::info! etc. will work. Simple way to get started.

</details>


### Q980. What is the purpose of EnvFilter?

- A) To filter by environment variable
- B) To filter spans/events by target (crate/module) and level (e.g., my_crate=debug, info)
- C) To set the log level
- D) To enable tracing

<details>
<summary>Show Answer</summary>

**Q980. B)** EnvFilter parses RUST_LOG-style filters: "crate=debug,other=info" means debug for crate, info for other. Used with with_env_filter on the subscriber. Controls verbosity per module.

</details>


### Q981. In tracing, what does Span::enter do?

- A) Exits the span
- B) Enters the span: creates a guard that, when dropped, exits the span. The span is the current context until the guard is dropped
- C) Creates the span
- D) Closes the span

<details>
<summary>Show Answer</summary>

**Q981. B)** span.enter() returns a Guard. While the guard is held, the span is the current span (events are associated with it). Dropping the guard exits the span. RAII for span scope.

</details>


### Q982. In axum, what is an Extractor?

- A) A middleware
- B) A type that extracts data from the request (path params, query, JSON body, etc.) and can be used as a handler parameter
- C) A router
- D) A response type

<details>
<summary>Show Answer</summary>

**Q982. B)** Extractors like Path, Query, Json, State are types you add as handler parameters. Axum extracts them from the request. E.g., fn handler(Path(id): Path<u32>, Json(body): Json<Body>) { ... }.

</details>


### Q983. What is Tower?

- A) A web framework
- B) A library of composable service abstractions (Service, Layer) used by hyper and axum for middleware and clients
- C) A database driver
- D) A templating engine

<details>
<summary>Show Answer</summary>

**Q983. B)** Tower provides Service (async request/response) and Layer (middleware). Hyper uses it; axum builds on it. Middleware like timeout, rate limit are Tower layers. Composable and reusable.

</details>


### Q984. In axum, how do you share state (e.g., a database pool) with handlers?

- A) Global static
- B) State<T>: add it to the router with .with_state(state), then use State<MyState> as an extractor in handlers
- C) Thread-local
- D) Request extensions only

<details>
<summary>Show Answer</summary>

**Q984. B)** Router::new().with_state(shared_state). State<T> extractor gives handlers access. T must be Clone (or Arc). Common for DbPool, Config, etc. Cloned per request, no locking for read-only.

</details>


### Q985. What is the relationship between hyper and axum?

- A) They are unrelated
- B) Axum is built on top of hyper; hyper provides the HTTP implementation, axum adds routing, extractors, etc.
- C) Hyper is built on axum
- D) They are the same

<details>
<summary>Show Answer</summary>

**Q985. B)** Hyper is the low-level HTTP library (client and server). Axum uses hyper as its transport and adds routing, extractors, middleware, and ergonomics. Tower sits between for middleware.

</details>


### Q986. In Tower, what does a Layer do?

- A) Handles requests
- B) Wraps a Service and returns a new Service (e.g., adding logging, timeout). Composable middleware
- C) Routes requests
- D) Parses bodies

<details>
<summary>Show Answer</summary>

**Q986. B)** Layer wraps a Service and returns another Service. E.g., TimeoutLayer wraps your handler and adds a timeout. Layers compose: layer1.layer(layer2).stack(). Used for middleware.

</details>


### Q987. What does axum's Json extractor do if the body is invalid JSON?

- A) Panics
- B) Returns 422 Unprocessable Entity with the parse error
- C) Returns 500
- D) Ignores the body

<details>
<summary>Show Answer</summary>

**Q987. B)** Json<T> deserializes the body. If it fails, axum returns 422 with the error. The handler is not called. You can customize with a custom extractor or rejection handling.

</details>


### Q988. In axum, how do you add middleware to a specific route?

- A) Middleware cannot be route-specific
- B) Use .layer() on the route or a group; layers apply to the routes they wrap
- C) Use #[middleware]
- D) Use route_middleware

<details>
<summary>Show Answer</summary>

**Q988. B)** router.route("/api", api_routes.layer(my_layer)) applies the layer only to /api. Or nest routes and apply layers to the nested router. Layer order matters (first added = outermost).

</details>


### Q989. What is the purpose of axum's Router?

- A) To parse JSON
- B) To compose routes and middleware; you build a tree of routes and merge/nest them
- C) To run the server
- D) To extract state

<details>
<summary>Show Answer</summary>

**Q989. B)** Router combines routes (get, post, etc.), nests routers, and applies layers. .merge() and .nest() compose routers. The final router is passed to axum::serve. Central abstraction for routing.

</details>


### Q990. In hyper, what is the role of the Service trait?

- A) To define routes
- B) To represent an async function from Request to Result<Response, Error>; the core abstraction for handlers and middleware
- C) To parse requests
- D) To manage connections

<details>
<summary>Show Answer</summary>

**Q990. B)** Service is async fn(Request) -> Result<Response, E>. Your handler implements it (via axum's IntoService or similar). Middleware wraps Services. Tower's core abstraction.

</details>


### Q991. What does #![no_std] do?

- A) Disables the standard library; the crate uses only core (and optionally alloc) for embedded or kernel code
- B) Removes all allocations
- C) Enables std
- D) Reduces binary size only

<details>
<summary>Show Answer</summary>

**Q991. A)** no_std means no std crate. You get core (and alloc if you use it). No threads, files, network, etc. Required for embedded, kernel, some WASM. You provide panic handler, allocator if needed.

</details>


### Q992. What is a target triple?

- A) A build flag
- B) A string identifying the target platform (e.g., x86_64-unknown-linux-gnu, thumbv7em-none-eabihf)
- C) A test
- D) A feature

<details>
<summary>Show Answer</summary>

**Q992. B)** Target triple: arch-vendor-os-abi. E.g., x86_64-unknown-linux-gnu, wasm32-unknown-unknown. rustup target add <triple>. cargo build --target <triple>. Identifies the compilation target.

</details>


### Q993. How do you conditionally use std only when available?

- A) You cannot
- B) #[cfg(feature = "std")] use std; and have a default "std" feature; use core when std is disabled
- C) Use std always
- D) Use #[no_std] only in release

<details>
<summary>Show Answer</summary>

**Q993. B)** Many crates have default = ["std"] and use std by default. With default-features = false, they use core. Code uses #[cfg(feature = "std")] for std-only code. Enables use in both std and no_std environments.

</details>


### Q994. What does wasm-bindgen primarily do?

- A) Compiles Rust to WASM
- B) Provides bindings between Rust and JavaScript: export Rust functions to JS, import JS in Rust, convert types
- C) Runs WASM
- D) Optimizes WASM

<details>
<summary>Show Answer</summary>

**Q994. B)** wasm-bindgen generates the glue: Rust fn -> JS callable, JS types -> Rust. #[wasm_bindgen] on functions/structs. Handles ABI, type conversion. rustc/LLVM does the actual WASM compilation.

</details>


### Q995. For embedded Rust, what does the cortex-m crate provide?

- A) A web framework
- B) Peripheral access, startup code, and utilities for ARM Cortex-M microcontrollers
- C) A HAL
- D) Only types

<details>
<summary>Show Answer</summary>

**Q995. B)** cortex-m provides low-level access: registers, interrupts, startup (reset handler, stack), delay. Often used with a HAL (hardware abstraction) for a specific chip. Foundation for embedded Rust on ARM.

</details>


### Q996. What is the purpose of [target.'cfg(...)'.dependencies] in Cargo.toml?

- A) To add features
- B) To add dependencies only when building for specific targets (e.g., different deps for wasm vs native)
- C) To patch
- D) To enable std

<details>
<summary>Show Answer</summary>

**Q996. B)** [target.'cfg(target_arch = "wasm32")'.dependencies] adds deps only for WASM. Useful when a crate has different implementations per target (e.g., wasm-bindgen for WASM, something else for native).

</details>


### Q997. In no_std, how do you allocate (e.g., Vec, String)?

- A) You cannot
- B) Use the alloc crate and a global allocator (e.g., alloc::alloc::Global); implement the allocator for your platform
- C) Use std
- D) Use Box only

<details>
<summary>Show Answer</summary>

**Q997. B)** alloc provides Vec, String, Box, etc. You need a global allocator (#[global_allocator]). On embedded, you might use a bump allocator or linked list allocator. core has no heap; alloc adds it.

</details>


### Q998. What does wasm32-unknown-unknown target produce?

- A) Native binary
- B) WebAssembly module that runs in browsers/Node without a JS runtime (no Emscripten)
- C) JavaScript
- D) Asm.js

<details>
<summary>Show Answer</summary>

**Q998. B)** wasm32-unknown-unknown: pure WASM. No libc, no OS. Runs in browser or wasmtime/wasmer. For Emscripten (JS glue, libc), use wasm32-unknown-emscripten. unknown-unknown is the common target.

</details>


### Q999. Why might an embedded crate use #[no_main]?

- A) To disable main
- B) The platform provides its own entry point (e.g., cortex-m's reset handler); the crate defines handlers, not main
- C) To enable main
- D) For testing

<details>
<summary>Show Answer</summary>

**Q999. B)** Embedded often has no main(). The MCU starts at a reset vector. cortex-m-rt provides the entry. Your code has #[entry] or interrupt handlers. #[no_main] says don't use the standard main.

</details>


### Q1000. What does the `critical-section` crate provide for no_std embedded code?

- A) A web framework
- B) A way to obtain a critical section (disable interrupts) for shared mutable state without a full RTOS
- C) Memory allocation
- D) Async support

<details>
<summary>Show Answer</summary>

**Q1000. B)** critical-section provides a way to run code with interrupts disabled. Used for shared mutable state in no_std. The implementation is platform-specific (cortex-m, etc.). Simpler than a full mutex for single-core embedded.

</details>
